<a href="https://colab.research.google.com/github/naylaakas/PembelajaranMesin-2026/blob/main/JS06/datasetPBL/Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install annoy hnswlib faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 6.9 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 19.1 MB/s  0:00:01
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=237d21a2ad92d1f58a533d90097a83439ea6289219b2378df6eda1dcf9e14376
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679796 sha256=e1ec78177b816e5bb1a0cb5e107976378b23a1e5e20d2eddec56377df89e0819
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50

In [5]:
import os
import time
import platform
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# Cek ketersediaan library ANN
try:
    from annoy import AnnoyIndex
    HAS_ANNOY = True
except ImportError:
    HAS_ANNOY = False

try:
    import hnswlib
    HAS_HNSW = True
except ImportError:
    HAS_HNSW = False

try:
    import faiss
    HAS_FAISS = True
except ImportError:
    HAS_FAISS = False

# -------------------------------
# 1. Deteksi otomatis lokasi dataset (Colab vs PyDroid3 Android)
# -------------------------------
if os.path.exists('cafes.csv') and os.path.exists('reviews.csv'):
    path_cafes = 'cafes.csv'
    path_reviews = 'reviews.csv'
elif os.path.exists('/storage/emulated/0/Download/cafes.csv'):
    path_cafes = '/storage/emulated/0/Download/cafes.csv'
    path_reviews = '/storage/emulated/0/Download/reviews.csv'
else:
    raise FileNotFoundError("File 'cafes.csv' dan 'reviews.csv' tidak ditemukan. Unggah ke Colab atau simpan di folder Download HP.")

df_cafes = pd.read_csv(path_cafes)
df_reviews = pd.read_csv(path_reviews)
df = pd.merge(df_reviews, df_cafes, on='cafe_id', how='left', suffixes=('_review', '_cafe'))

# Bentuk matriks fitur d = 128 (5 fitur numerik + 123 fitur teks TF-IDF)
d = 128
tfidf = TfidfVectorizer(max_features=123, ngram_range=(1, 2))
X_txt = tfidf.fit_transform((df['name'].fillna('') + " " + df['text'].fillna(''))).toarray()
X_num = df[['rating_review', 'rating_cafe', 'review_count', 'latitude', 'longitude']].fillna(0).values

X = np.hstack([X_num, X_txt]).astype('float32')
n_samples = X.shape[0]

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype('float32')
k = 10  # jumlah nearest neighbors

print(f"Environment : {platform.system()} | Arsitektur: {platform.machine()} | CPU Cores: {os.cpu_count()}")
print(f"Dataset     : {X_scaled.shape} (dari '{path_cafes}')\n")

# -------------------------------
# 2. Exact NN (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# 3. Annoy
# -------------------------------
time_annoy = None
if HAS_ANNOY:
    start = time.time()
    f = X_scaled.shape[1]
    index_annoy = AnnoyIndex(f, 'euclidean')
    for i, v in enumerate(X_scaled):
        index_annoy.add_item(i, v)
    index_annoy.build(10)
    idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
    time_annoy = time.time() - start
    print(f"Annoy done in {time_annoy:.3f} s")
else:
    print("Annoy tidak terpasang.")

# -------------------------------
# 4. HNSW
# -------------------------------
time_hnsw = None
if HAS_HNSW:
    start = time.time()
    p_hnsw = hnswlib.Index(space='l2', dim=d)
    p_hnsw.init_index(max_elements=n_samples, ef_construction=200, M=16)
    p_hnsw.add_items(X_scaled)
    p_hnsw.set_ef(200)
    idx_hnsw, _ = p_hnsw.knn_query(X_scaled, k=k)
    time_hnsw = time.time() - start
    print(f"HNSW done in {time_hnsw:.3f} s")
else:
    print("HNSW tidak terpasang.")

# -------------------------------
# 5. FAISS IVF
# -------------------------------
time_faiss = None
if HAS_FAISS:
    start = time.time()
    quantizer = faiss.IndexFlatL2(d)
    index_faiss = faiss.IndexIVFFlat(quantizer, d, 50, faiss.METRIC_L2)
    index_faiss.train(X_scaled)
    index_faiss.add(X_scaled)
    index_faiss.nprobe = 10
    _, idx_faiss = index_faiss.search(X_scaled, k)
    time_faiss = time.time() - start
    print(f"FAISS IVF done in {time_faiss:.3f} s")
else:
    print("FAISS tidak terpasang / tidak didukung di arsitektur perangkat ini.")

# -------------------------------
# 6. Tampilkan ringkasan waktu
# -------------------------------
print("\n=== Ringkasan Waktu (detik) ===")
print(f"Exact NN : {time_exact:.3f}")
print(f"Annoy    : {f'{time_annoy:.3f}' if time_annoy is not None else 'N/A'}")
print(f"HNSW     : {f'{time_hnsw:.3f}' if time_hnsw is not None else 'N/A'}")
print(f"FAISS    : {f'{time_faiss:.3f}' if time_faiss is not None else 'N/A'}")

Environment : Linux | Arsitektur: x86_64 | CPU Cores: 2
Dataset     : (2230, 128) (dari 'cafes.csv')

Exact NN done in 0.142 s
Annoy done in 0.214 s
HNSW done in 1.100 s
FAISS IVF done in 0.441 s

=== Ringkasan Waktu (detik) ===
Exact NN : 0.142
Annoy    : 0.214
HNSW     : 1.100
FAISS    : 0.441
